In [3]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

In [4]:
#import dataset
df = pd.read_csv("student_data.csv", sep=';')

In [5]:
#pisahkan fitur dan target
X = df.drop('Target', axis=1)
y = df['Target']

In [6]:
#encoding target
le = LabelEncoder()
y_encoded = le.fit_transform(y)

y_cat = to_categorical(y_encoded, num_classes=3)

In [7]:
#encoding fitur kategorikal
X_encoded = pd.get_dummies(X, drop_first=True)

In [8]:
#feature scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_encoded)

In [9]:
#train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_cat,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

In [10]:
#one-hot target
y_train_cat = to_categorical(y_train, 3)
y_test_cat  = to_categorical(y_test, 3)

In [11]:
!pip install torchdiffeq

In [12]:
import torch
import torch.nn as nn
from torchdiffeq import odeint

In [13]:
class ODEFunc(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, 64),
            nn.Tanh(),
            nn.Linear(64, dim)
        )

    def forward(self, t, x):
        return self.net(x)

In [14]:
class NeuralODE(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.func = ODEFunc(input_dim)
        self.classifier = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        t = torch.tensor([0, 1]).float()
        out = odeint(self.func, x, t)
        out = out[1]
        return self.classifier(out)

In [15]:
X_train_torch = torch.tensor(X_train, dtype=torch.float32)
X_test_torch  = torch.tensor(X_test, dtype=torch.float32)

y_train_torch = torch.tensor(y_train, dtype=torch.long)
y_test_torch  = torch.tensor(y_test, dtype=torch.long)

In [16]:
model = NeuralODE(X_train.shape[1], 3)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [19]:
epochs = 30

for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(X_train_torch)
    loss = criterion(outputs, y_train_torch)
    loss.backward()
    optimizer.step()

    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

RuntimeError: Expected floating point type for target with class probabilities, got Long

In [ ]:
with torch.no_grad():
    preds = model(X_test_torch)
    _, predicted = torch.max(preds, 1)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, predicted.numpy()))